# Gorilla reproduction: TF-IDF retrieval
Kubernetes pilot: rank API **descriptions only** with TF-IDF and cosine similarity. Evaluate top-1/top-3 accuracy using the manually verified labels in `data/evaluation/answers.json`; labels never enter retrieval.

Dependencies: `pip install scikit-learn ipykernel`. Run all cells from the project root or `notebooks/`. This is a retrieval baseline, not the full paper experiment. Generation compares Google AI Studio and a local vLLM endpoint. Cluster inspection is read-only; generated Kubernetes calls are never executed.

In [1]:
import json
import hashlib
from pathlib import Path

import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/apis.json").is_file()), None)
assert root is not None, "Run from the project root or notebooks directory."

def load_json(relative_path):
    return json.loads((root / relative_path).read_text(encoding="utf-8"))

apis = load_json("data/apis.json")["apis"]
queries = load_json("data/queries.json")["requests"]
api_ids = [api["api_name"] for api in apis]
query_ids = [query["request_id"] for query in queries]
assert len(api_ids) == len(set(api_ids)) >= 3
assert len(query_ids) == len(set(query_ids)) > 0
assert all(set(query) == {"request_id", "input"} for query in queries)
print(f"APIs: {len(apis)} | Queries: {len(queries)} | scikit-learn: {sklearn.__version__}")

APIs: 30 | Queries: 20 | scikit-learn: 1.9.1


Fit vocabulary and IDF on catalog descriptions only. Transform queries without refitting. Use English stop words and word unigrams/bigrams; cosine ties follow catalog order.

In [2]:
documents = [api["description"] for api in apis]
query_texts = [query["input"] for query in queries]
assert all(isinstance(text, str) and text.strip() for text in documents + query_texts)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
api_vectors = vectorizer.fit_transform(documents)
query_vectors = vectorizer.transform(query_texts)
scores = cosine_similarity(query_vectors, api_vectors)
ranked_indices = np.argsort(-scores, axis=1, kind="stable")
top3 = np.asarray(api_ids)[ranked_indices[:, :3]]
assert np.all(np.isfinite(scores))
assert np.all(query_vectors.getnnz(axis=1) > 0), "A query has no catalog vocabulary overlap."

Join gold labels by `request_id` after ranking. Top-k accuracy is the fraction of queries whose correct API appears among the first k results. `results` retains per-query predictions for inspection.

In [3]:
answers = load_json("data/evaluation/answers.json")["answers"]
gold = {answer["request_id"]: answer["correct_api"] for answer in answers}
assert len(gold) == len(answers), "Duplicate answer IDs."
assert set(gold) == set(query_ids), "Query/answer IDs differ."
assert set(gold.values()) <= set(api_ids), "A gold API is absent from the catalog."
expected = np.asarray([gold[request_id] for request_id in query_ids])

hits_at_1 = top3[:, 0] == expected
hits_at_3 = (top3 == expected[:, None]).any(axis=1)
assert np.all(hits_at_1 <= hits_at_3)
for k, hits in [(1, hits_at_1), (3, hits_at_3)]:
    print(f"Top-{k} accuracy: {hits.mean():.1%} ({hits.sum()}/{len(hits)})")

results = [
    {"request_id": request_id, "correct_api": gold[request_id],
     "top_3": top3[i].tolist(), "hit_at_1": bool(hits_at_1[i]),
     "hit_at_3": bool(hits_at_3[i])}
    for i, request_id in enumerate(query_ids)
]
dataset_sha256 = {
    path: hashlib.sha256((root / path).read_bytes()).hexdigest()
    for path in ["data/apis.json", "data/queries.json", "data/evaluation/answers.json"]
}

Top-1 accuracy: 50.0% (10/20)
Top-3 accuracy: 85.0% (17/20)


These are descriptive scores on this small, fixed pilot set. No parameter tuning or held-out generalization claim is made.

## Google model access and choice
Read `credentials` only in memory and send the key in a header to Google's fixed HTTPS endpoint. Never print keys, raw API errors, or credential contents. The checks below use a tiny generic prompt—not benchmark queries.

Verified prices/access are saved in `data/evaluation/model_access.json`. Set `REFRESH_ACCESS = True` only to repeat the tiny access probes; the default displays the saved evidence.

In [4]:
import re
import urllib.request
import urllib.error
import urllib.parse

class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *args, **kwargs):
        return None

def read_google_key():
    # Supports a raw key, a provider-label/key pair, or a standard JSON/env field.
    try:
        raw = (root / "credentials").read_text(encoding="utf-8").strip()
        names = {"api_key", "google_api_key", "gemini_api_key", "key"}
        try:
            data = json.loads(raw)
        except ValueError:
            data = None
        candidates = []
        if isinstance(data, dict):
            candidates = [v for k, v in data.items()
                          if k.lower() in names and isinstance(v, str)]
        else:
            lines = [line.strip().strip("`\"'") for line in raw.splitlines() if line.strip()]
            if len(lines) == 2 and re.search(r"google|gemini|ai.?studio", lines[0], re.I):
                candidates = [lines[1]]
            elif len(lines) == 1:
                match = re.fullmatch(r"(?:export\s+)?([A-Za-z_]+)\s*[:=]\s*(.+)", lines[0])
                candidates = ([match[2].strip().strip("\"'")]
                              if match and match[1].lower() in names else lines)
        candidates = list(set(candidates))
        if len(candidates) == 1 and re.fullmatch(r"[A-Za-z0-9._~+=/-]{16,4096}", candidates[0]):
            return candidates[0]
    except Exception:
        pass
    raise RuntimeError("Unable to read one Google key; credential details suppressed.") from None

def google_request(endpoint, payload=None):
    if not re.fullmatch(r"models(?:/[A-Za-z0-9._-]+:generateContent)?(?:\?pageToken=[A-Za-z0-9%._~-]+)?", endpoint):
        return {"ok": False, "error": "INVALID_ENDPOINT"}
    try:
        request = urllib.request.Request(
            "https://generativelanguage.googleapis.com/v1beta/" + endpoint,
            data=None if payload is None else json.dumps(payload).encode(),
            headers={"x-goog-api-key": read_google_key(), "Content-Type": "application/json"},
        )
        with urllib.request.build_opener(NoRedirect).open(request, timeout=60) as response:
            return {"ok": True, "http_status": response.status, "data": json.load(response)}
    except urllib.error.HTTPError as error:
        return {"ok": False, "error": "HTTP_ERROR", "http_status": error.code}
    except Exception:
        return {"ok": False, "error": "CREDENTIAL_NETWORK_OR_RESPONSE_ERROR"}

# Validate the file without displaying or keeping a key in notebook output.
assert bool(read_google_key())
print("Credential loaded; value hidden.")

Credential loaded; value hidden.


In [5]:
REFRESH_ACCESS = False
access_report = load_json("data/evaluation/model_access.json")

if REFRESH_ACCESS:
    available = set()
    endpoint = "models"
    while True:
        response = google_request(endpoint)
        if not response["ok"]:
            raise RuntimeError("Model discovery failed; raw error suppressed.")
        available.update(m["name"] for m in response["data"].get("models", [])
                         if "generateContent" in m.get("supportedGenerationMethods", []))
        token = response["data"].get("nextPageToken")
        if not token:
            break
        endpoint = "models?pageToken=" + urllib.parse.quote(token, safe="")
    for option in access_report["models"]:
        option.update(access_status="NOT_LISTED", temperature_zero_verified=False, http_status=None)
        if "models/" + option["model"] not in available:
            continue
        probe = google_request("models/" + option["model"] + ":generateContent", {
            "contents": [{"role": "user", "parts": [{"text": "Reply with the single word OK."}]}],
            "generationConfig": {"temperature": 0, "maxOutputTokens": 256},
        })
        has_text = probe["ok"] and any(
            part.get("text") and not part.get("thought")
            for candidate in probe.get("data", {}).get("candidates", [])
            for part in candidate.get("content", {}).get("parts", [])
        )
        option.update(access_status="PASS" if has_text else "API_ERROR",
                      temperature_zero_verified=probe["ok"],
                      http_status=probe.get("http_status"))
    print("Fresh generic probes completed; pricing remains the dated snapshot below.")
else:
    print("Saved access checks:", access_report["checked_on"])

print("Model | Access | Temperature 0 | Paid input/output USD per 1M tokens")
for option in access_report["models"]:
    price = ("Free tier only" if option["paid_input_usd_per_million"] is None else
             f"${option['paid_input_usd_per_million']:.2f} / ${option['paid_output_usd_per_million']:.2f}")
    print(f"{option['model']} | {option['access_status']} | "
          f"{option['temperature_zero_verified']} | {price}")
print("Free tiers have quotas; actual account billing tier is not inferred.")

Saved access checks: 2026-10-28
Model | Access | Temperature 0 | Paid input/output USD per 1M tokens
gemma-4-26b-a4b-it | PASS | True | Free tier only
gemma-4-31b-it | PASS | True | Free tier only
gemini-2.5-flash-lite | API_ERROR | False | $0.10 / $0.40
gemini-3.1-flash-lite | PASS | True | $0.25 / $1.50
Free tiers have quotas; actual account billing tier is not inferred.


## Generate one call per query
Selected: **`gemma-4-31b-it`**, temperature **0**, output limit 4096 tokens. Each prompt contains only the query and its three retrieved descriptions, plus catalog-derived SDK method/argument metadata. Gold labels and reference calls are never sent.

Calls use preconfigured SDK client aliases (`core_v1`, `apps_v1`, etc.). PATCH aliases `_merge`, `_strategic`, and `_json` select Content-Type through `ApiClient.default_headers`; no clients are instantiated. [SDK source](https://github.com/kubernetes-client/python/blob/v35.0.0/kubernetes/client/api_client.py).

The default reuses saved responses. Set `RUN_BENCHMARK=True` to send missing requests; errors are retained without silent retries.

In [6]:
import sys
sys.path.insert(0, str(root / "notebooks"))
from gorilla_eval import run_generation, evaluate_run

SELECTED_MODEL = "gemma-4-31b-it"
RUN_BENCHMARK = False  # Cached evaluation by default; True permits missing model requests.
generation_file = root / "data/evaluation/gemma_4_31b_generation.json"
run = run_generation(queries, top3.tolist(), apis, SELECTED_MODEL,
                     google_request, generation_file, allow_requests=RUN_BENCHMARK)
print(f"Model: {run['model']} | Responses: {len(run['records'])} | Temperature: 0")

Model: gemma-4-31b-it | Responses: 20 | Temperature: 0


## AST evaluation — no code execution
Parse one call with literal keyword arguments or `NO_MATCHING_API`. Check the canonical function, retrieved-candidate membership, path/query arguments, complete reference body, and PATCH Content-Type. Dictionary order is ignored; list order and value types matter. Omitted `watch=False`/`follow=False` and redundant body namespace are accepted. Other literal differences fail; this measures reference-call agreement, not Kubernetes runtime success.

Correct-call rate = correct function **and** arguments / all 20 queries. Also report the rate among API-successful requests. API errors, incomplete output, invalid calls, and abstentions remain separate. An abstention is never a correct call, even when retrieval missed the gold API.

In [7]:
evaluation = evaluate_run(run, answers, apis)
evaluation["dataset_sha256"] = dataset_sha256
evaluation["evaluator_sha256"] = hashlib.sha256((root / "notebooks/gorilla_eval.py").read_bytes()).hexdigest()
evaluation["generation_sha256"] = hashlib.sha256(generation_file.read_bytes()).hexdigest()
metrics = evaluation["summary"]
print(f"Correct-call rate: {metrics['correct_call_rate_all_queries']:.1%} "
      f"({metrics['correct_calls']}/{metrics['queries']})")
conditional = metrics["correct_call_rate_api_successes"]
print("Among API-successful requests:", "N/A" if conditional is None else f"{conditional:.1%}")
print(f"Function matches: {metrics['function_correct']}/{metrics['queries']}")
print(f"Model API errors: {metrics['api_errors']}/{metrics['queries']}")
print("Outcome counts:", metrics["status_counts"])
(root / "data/evaluation/gemma_4_31b_evaluation.json").write_text(
    json.dumps(evaluation, indent=2) + "\n", encoding="utf-8")
# Per-query details are available locally in evaluation["rows"].

Correct-call rate: 15.0% (3/20)
Among API-successful requests: 75.0%
Function matches: 4/20
Model API errors: 16/20
Outcome counts: {'API_ERROR': 16, 'CORRECT_CALL': 3, 'WRONG_ARGUMENTS': 1}


5868

## Qwen3-8B on vLLM
The existing Pod and served model passed readiness, health, and temperature-0 checks. Private connection coordinates live in ignored `notebooks/vllm_connection.local.json`; Google credentials are never sent to vLLM.

Use the **same prompts, top-three APIs, and AST scorer** as Gemma. Temperature is 0; Qwen thinking is disabled for call-only output. Before inference, tokenize all unchanged prompts and choose one output limit that fits every prompt (up to 4096 tokens). The effective limit is recorded; Gemma's settings/results remain unchanged. [Qwen's vLLM documentation](https://qwen.readthedocs.io/en/latest/deployment/vllm.html).

Cached responses are reused by default. `RUN_QWEN=True` sends missing requests through a temporary loopback tunnel; it does not change Kubernetes resources.

In [8]:
from gorilla_vllm import run_qwen, verify_same_inputs

RUN_QWEN = False
qwen_file = root / "data/evaluation/qwen3_8b_generation.json"
qwen_run = run_qwen(queries, top3.tolist(), apis,
                    root / "notebooks/vllm_connection.local.json", qwen_file,
                    allow_requests=RUN_QWEN)
verify_same_inputs(run, qwen_run)
qwen_evaluation = evaluate_run(qwen_run, answers, apis)
qwen_evaluation["dataset_sha256"] = dataset_sha256
qwen_evaluation["evaluator_sha256"] = hashlib.sha256((root / "notebooks/gorilla_eval.py").read_bytes()).hexdigest()
qwen_evaluation["transport_sha256"] = hashlib.sha256((root / "notebooks/gorilla_vllm.py").read_bytes()).hexdigest()
qwen_evaluation["generation_sha256"] = hashlib.sha256(qwen_file.read_bytes()).hexdigest()
(root / "data/evaluation/qwen3_8b_evaluation.json").write_text(
    json.dumps(qwen_evaluation, indent=2) + "\n", encoding="utf-8")
print("Qwen output-token limit:", qwen_run["generation_config"]["max_tokens"])
print("Identical prompts and retrieved candidates: verified")
print("Qwen outcomes:", qwen_evaluation["summary"]["status_counts"])

Qwen output-token limit: 3310
Identical prompts and retrieved candidates: verified
Qwen outcomes: {'CORRECT_CALL': 4, 'NO_MATCHING_API': 1, 'WRONG_ARGUMENTS': 6, 'INVALID_OUTPUT': 8, 'WRONG_FUNCTION': 1}


## Model comparison
Overall correct-call rate includes every query; the conditional rate excludes model API/transport failures. Interpret these as results of these two runs: Gemma had substantial API failures, and the models use different serving/reasoning configurations. No generated code is executed.

An initial vLLM connection run was excluded after a local tunnel log-buffer bug caused timeouts. All 20 prompts were rerun after fixing log draining; the affected responses remain in `data/evaluation/qwen3_8b_transport_failure_run.json` and the diagnosis in `qwen3_8b_transport_fix.json`.

In [9]:
from IPython.display import Markdown, display

comparison = []
for result in [evaluation, qwen_evaluation]:
    m = result["summary"]
    comparison.append({"model": result["model"], **m})
lines = ["| Model | Correct calls / 20 | Among API successes | Function matches | API errors |",
         "|---|---:|---:|---:|---:|"]
for row in comparison:
    conditional = row["correct_call_rate_api_successes"]
    rate = "N/A" if conditional is None else f"{conditional:.1%} ({row['correct_calls']}/{row['api_successes']})"
    lines.append(f"| {row['model']} | {row['correct_call_rate_all_queries']:.1%} "
                 f"({row['correct_calls']}/{row['queries']}) | {rate} | "
                 f"{row['function_correct']}/{row['queries']} | {row['api_errors']}/{row['queries']} |")
display(Markdown("\n".join(lines)))
(root / "data/evaluation/model_comparison.json").write_text(
    json.dumps({"same_prompts_and_top3": True, "generated_code_executed": False,
                "models": comparison,
                "configurations": {run["model"]: run["generation_config"],
                                   qwen_run["model"]: qwen_run["generation_config"]}}, indent=2) + "\n",
    encoding="utf-8")

| Model | Correct calls / 20 | Among API successes | Function matches | API errors |
|---|---:|---:|---:|---:|
| gemma-4-31b-it | 15.0% (3/20) | 75.0% (3/4) | 4/20 | 16/20 |
| Qwen3-8B | 20.0% (4/20) | 20.0% (4/20) | 10/20 | 0/20 |

1161